# Iris Classification with Perceptron Learning

Энэ notebook нь `Iris.csv` өгөгдөл дээр Perceptron learning алгоритмыг бэлэн машин сургалтын сан ашиглалгүйгээр хэрэгжүүлнэ. Өгөгдлийг 70:30 харьцаагаар `Training set` ба `Test set` болгон хувааж, сургалтын өгөгдөл дээр загвар сургаад туршилтын өгөгдөл дээр үнэлнэ.

In [1]:
import csv
import random
from collections import defaultdict
from pathlib import Path

In [2]:
DATA_FILE = Path("Iris.csv")
TRAIN_RATIO = 0.7
SEED = 42
EPOCHS = 25
LEARNING_RATE = 0.1

In [3]:
def load_iris_dataset(path):
    dataset = []
    with path.open(newline="", encoding="utf-8-sig") as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            dataset.append(
                {
                    "features": [
                        float(row["SepalLengthCm"]),
                        float(row["SepalWidthCm"]),
                        float(row["PetalLengthCm"]),
                        float(row["PetalWidthCm"]),
                    ],
                    "label": row["Species"],
                }
            )
    return dataset


def stratified_split(dataset, train_ratio=TRAIN_RATIO, seed=SEED):
    rng = random.Random(seed)
    grouped = defaultdict(list)

    for sample in dataset:
        grouped[sample["label"]].append(sample)

    train_set = []
    test_set = []

    for label_samples in grouped.values():
        shuffled = label_samples[:]
        rng.shuffle(shuffled)
        split_index = int(len(shuffled) * train_ratio)
        train_set.extend(shuffled[:split_index])
        test_set.extend(shuffled[split_index:])

    rng.shuffle(train_set)
    rng.shuffle(test_set)
    return train_set, test_set


def fit_standardizer(dataset):
    feature_count = len(dataset[0]["features"])
    means = [0.0] * feature_count
    stds = [0.0] * feature_count

    for sample in dataset:
        for index, value in enumerate(sample["features"]):
            means[index] += value

    for index in range(feature_count):
        means[index] /= len(dataset)

    for sample in dataset:
        for index, value in enumerate(sample["features"]):
            stds[index] += (value - means[index]) ** 2

    for index in range(feature_count):
        stds[index] = (stds[index] / len(dataset)) ** 0.5
        if stds[index] == 0:
            stds[index] = 1.0

    return means, stds


def standardize_dataset(dataset, means, stds):
    standardized = []
    for sample in dataset:
        standardized.append(
            {
                "features": [
                    (value - means[index]) / stds[index]
                    for index, value in enumerate(sample["features"])
                ],
                "label": sample["label"],
            }
        )
    return standardized

In [4]:
class OneVsRestPerceptron:
    def __init__(self, labels, feature_count, learning_rate=LEARNING_RATE, epochs=EPOCHS):
        self.labels = list(labels)
        self.feature_count = feature_count
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = {label: [0.0] * feature_count for label in self.labels}
        self.biases = {label: 0.0 for label in self.labels}

    def score(self, label, features):
        return sum(weight * value for weight, value in zip(self.weights[label], features)) + self.biases[label]

    def predict(self, features):
        best_label = self.labels[0]
        best_score = self.score(best_label, features)

        for label in self.labels[1:]:
            current_score = self.score(label, features)
            if current_score > best_score:
                best_score = current_score
                best_label = label

        return best_label

    def fit(self, dataset):
        history = []

        for epoch in range(1, self.epochs + 1):
            mistakes = 0

            for sample in dataset:
                features = sample["features"]
                actual_label = sample["label"]
                predicted_label = self.predict(features)

                if predicted_label != actual_label:
                    mistakes += 1

                    for index, value in enumerate(features):
                        self.weights[actual_label][index] += self.learning_rate * value
                        self.weights[predicted_label][index] -= self.learning_rate * value

                    self.biases[actual_label] += self.learning_rate
                    self.biases[predicted_label] -= self.learning_rate

            history.append({"epoch": epoch, "mistakes": mistakes})

            if mistakes == 0:
                break

        return history

In [5]:
def evaluate(model, dataset, labels):
    predictions = []
    correct = 0
    confusion = {actual: {predicted: 0 for predicted in labels} for actual in labels}

    for sample in dataset:
        predicted = model.predict(sample["features"])
        actual = sample["label"]
        predictions.append((actual, predicted))
        confusion[actual][predicted] += 1

        if predicted == actual:
            correct += 1

    accuracy = correct / len(dataset)
    return accuracy, confusion, predictions


def print_confusion_matrix(confusion, labels):
    print("Confusion matrix")
    header = "Actual \\ Pred".ljust(20) + "".join(label.ljust(18) for label in labels)
    print(header)
    for actual in labels:
        row = actual.ljust(20)
        for predicted in labels:
            row += str(confusion[actual][predicted]).ljust(18)
        print(row)

In [6]:
dataset = load_iris_dataset(DATA_FILE)
train_set, test_set = stratified_split(dataset)

print("Dataset split (70:30)")
print("  Training samples:", len(train_set))
print("  Test samples    :", len(test_set))

train_counts = defaultdict(int)
test_counts = defaultdict(int)

for sample in train_set:
    train_counts[sample["label"]] += 1
for sample in test_set:
    test_counts[sample["label"]] += 1

print("\nTraining class counts:")
for label in sorted(train_counts):
    print(f"  {label}: {train_counts[label]}")

print("\nTest class counts:")
for label in sorted(test_counts):
    print(f"  {label}: {test_counts[label]}")

Dataset split (70:30)
  Training samples: 105
  Test samples    : 45

Training class counts:
  Iris-setosa: 35
  Iris-versicolor: 35
  Iris-virginica: 35

Test class counts:
  Iris-setosa: 15
  Iris-versicolor: 15
  Iris-virginica: 15


In [7]:
means, stds = fit_standardizer(train_set)
train_standardized = standardize_dataset(train_set, means, stds)
test_standardized = standardize_dataset(test_set, means, stds)

labels = sorted({sample["label"] for sample in dataset})
model = OneVsRestPerceptron(labels, feature_count=4)
history = model.fit(train_standardized)

print("Training progress")
for item in history:
    print(f"  Epoch {item['epoch']:2d}: mistakes = {item['mistakes']}")

Training progress
  Epoch  1: mistakes = 24
  Epoch  2: mistakes = 15
  Epoch  3: mistakes = 5
  Epoch  4: mistakes = 4
  Epoch  5: mistakes = 10
  Epoch  6: mistakes = 7
  Epoch  7: mistakes = 8
  Epoch  8: mistakes = 5
  Epoch  9: mistakes = 10
  Epoch 10: mistakes = 7
  Epoch 11: mistakes = 7
  Epoch 12: mistakes = 4
  Epoch 13: mistakes = 7
  Epoch 14: mistakes = 6
  Epoch 15: mistakes = 2
  Epoch 16: mistakes = 2
  Epoch 17: mistakes = 2
  Epoch 18: mistakes = 2
  Epoch 19: mistakes = 2
  Epoch 20: mistakes = 5
  Epoch 21: mistakes = 9
  Epoch 22: mistakes = 6
  Epoch 23: mistakes = 5
  Epoch 24: mistakes = 4
  Epoch 25: mistakes = 7


In [8]:
accuracy, confusion, predictions = evaluate(model, test_standardized, labels)

print("Learned parameters")
for label in labels:
    rounded_weights = [round(weight, 4) for weight in model.weights[label]]
    print(f"  {label}")
    print(f"    weights: {rounded_weights}")
    print(f"    bias   : {round(model.biases[label], 4)}")

print()
print_confusion_matrix(confusion, labels)
print()
print(f"Test accuracy: {accuracy * 100:.2f}%")

Learned parameters
  Iris-setosa
    weights: [-0.3622, 0.2903, -0.6332, -0.6603]
    bias   : -0.1
  Iris-versicolor
    weights: [0.2718, -0.0549, -0.2897, -0.3315]
    bias   : 1.0
  Iris-virginica
    weights: [0.0903, -0.2354, 0.9229, 0.9918]
    bias   : -0.9

Confusion matrix
Actual \ Pred       Iris-setosa       Iris-versicolor   Iris-virginica    
Iris-setosa         15                0                 0                 
Iris-versicolor     0                 15                0                 
Iris-virginica      0                 1                 14                

Test accuracy: 97.78%


In [9]:
print("Sample test predictions")
for index, (actual, predicted) in enumerate(predictions[:10], start=1):
    print(f"  {index:2d}. actual = {actual:<15} predicted = {predicted}")

Sample test predictions
   1. actual = Iris-virginica  predicted = Iris-virginica
   2. actual = Iris-setosa     predicted = Iris-setosa
   3. actual = Iris-versicolor predicted = Iris-versicolor
   4. actual = Iris-versicolor predicted = Iris-versicolor
   5. actual = Iris-versicolor predicted = Iris-versicolor
   6. actual = Iris-setosa     predicted = Iris-setosa
   7. actual = Iris-virginica  predicted = Iris-virginica
   8. actual = Iris-setosa     predicted = Iris-setosa
   9. actual = Iris-setosa     predicted = Iris-setosa
  10. actual = Iris-versicolor predicted = Iris-versicolor
